# SWIFT Data Extraction
## Notebook 01 — Extract currency shares from SWIFT Global Currency Tracker PDFs

**Purpose:** Extract monthly global payment currency shares from all SWIFT RMB/Global Currency Tracker PDF reports and compile into a single CSV for HHI analysis.

**Data source:** SWIFT Global Currency Tracker (formerly RMB Tracker), October 2022 – March 2026
**Output files:**
- SWIFT_Global_Currency_Shares.csv
- RMB_Evolution_2019_2022.csv
**Date:** March 2026

In [3]:
import zipfile
import xml.etree.ElementTree as ET
import csv
from datetime import datetime, timedelta

def excel_date(serial):
    if serial > 59:
        serial -= 1
    dt = datetime(1899, 12, 31) + timedelta(days=int(serial))
    return dt.strftime('%Y-%m')

input_path = "/Users/psat0501/Desktop/HSG/Master Thesis/Thesis Data/SWIFT Dataset/October 22 - Evolution of RMB's share.csv"
output_path = "/Users/psat0501/Desktop/HSG/Master Thesis/Thesis Data/SWIFT Dataset/RMB_Evolution_2019_2022.csv"

rows = []
with zipfile.ZipFile(input_path) as z:
    ss_xml = z.read('xl/sharedStrings.xml')
    ss_tree = ET.fromstring(ss_xml)
    ns = {'ns': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    strings = [si.find('.//ns:t', ns).text 
               for si in ss_tree.findall('ns:si', ns)]
    sheet_xml = z.read('xl/worksheets/sheet1.xml')
    sheet_tree = ET.fromstring(sheet_xml)
    data_rows = sheet_tree.findall('.//ns:row', ns)

    for row in data_rows[2:]:
        cells = []
        for c in row.findall('ns:c', ns):
            t = c.get('t')
            v = c.find('ns:v', ns)
            if v is not None:
                cells.append(strings[int(v.text)] if t == 's' else v.text)
            else:
                cells.append('')
        if len(cells) >= 2 and cells[0]:
            try:
                date = excel_date(float(cells[0]))
                share = round(float(cells[1]) * 100, 2)
                ranking = cells[2] if len(cells) > 2 else ''
                rows.append({
                    'Date': date,
                    'RMB_Share_Pct': share,
                    'RMB_Ranking': ranking,
                    'Source': 'SWIFT RMB Tracker Oct 2022 rolling chart'
                })
            except:
                pass

with open(output_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, 
                fieldnames=['Date','RMB_Share_Pct','RMB_Ranking','Source'])
    writer.writeheader()
    writer.writerows(rows)

print(f"Converted {len(rows)} rows")
print(f"Date range: {rows[0]['Date']} to {rows[-1]['Date']}")
print(f"Saved to: {output_path}")

Converted 45 rows
Date range: 2019-01 to 2022-09
Saved to: /Users/psat0501/Desktop/HSG/Master Thesis/Thesis Data/SWIFT Dataset/RMB_Evolution_2019_2022.csv


In [6]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "pdfplumber"], check=True)

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', '-m', 'pip', 'install', 'pdfplumber'], returncode=0)

In [8]:
import pdfplumber, re, csv
from pathlib import Path

SWIFT_FOLDER = "/Users/psat0501/Desktop/HSG/Master Thesis/Thesis Data/SWIFT Dataset"
OUTPUT = SWIFT_FOLDER + "/SWIFT_RMB_Monthly_Share.csv"

MONTHS = {
    "january":"01","february":"02","march":"03","april":"04",
    "may":"05","june":"06","july":"07","august":"08",
    "september":"09","october":"10","november":"11","december":"12"
}

def get_month(filename):
    fname = filename.lower()
    for m, num in MONTHS.items():
        if m in fname:
            yr = re.search(r'(20\d{2})', fname)
            if yr:
                return f"{yr.group(1)}-{num}"
    return None

rows = []
for pdf_path in sorted(Path(SWIFT_FOLDER).glob("*.pdf")):
    month = get_month(pdf_path.stem)
    if not month:
        continue
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text() or ""
                # Evolution page contains monthly RMB percentages
                if "evolution" in text.lower() and "rmb" in text.lower():
                    # Extract all percentages in RMB range (1.0 to 5.5%)
                    vals = [float(v) for v in 
                            re.findall(r'(\d+\.\d+)%', text)
                            if 1.0 <= float(v) <= 5.5]
                    if vals:
                        # The last value on the page is the most recent month
                        rows.append({
                            'Report_Month': month,
                            'RMB_Share_Pct': vals[-1],
                            'All_Values_Found': str(vals)
                        })
                        print(f"{month}: RMB = {vals[-1]}% ({len(vals)} values found)")
                        break
    except Exception as e:
        print(f"ERROR {pdf_path.name}: {e}")

rows.sort(key=lambda x: x['Report_Month'])
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['Report_Month','RMB_Share_Pct','All_Values_Found'])
    writer.writeheader()
    writer.writerows(rows)

print(f"\nSaved {len(rows)} rows to {OUTPUT}")

2026-03: RMB = 1.0% (90 values found)
2023-01: RMB = 1.0% (69 values found)
2024-04: RMB = 1.0% (77 values found)
2025-04: RMB = 1.0% (89 values found)
2023-08: RMB = 1.0% (65 values found)
2024-08: RMB = 1.0% (81 values found)
2025-08: RMB = 1.0% (87 values found)
2023-12: RMB = 1.0% (73 values found)
2024-12: RMB = 1.0% (85 values found)
2025-12: RMB = 1.0% (91 values found)
2023-02: RMB = 1.0% (70 values found)
2024-02: RMB = 1.0% (75 values found)
2025-02: RMB = 1.0% (87 values found)
2026-02: RMB = 1.0% (93 values found)
2024-01: RMB = 1.0% (74 values found)
2025-01: RMB = 1.0% (86 values found)
2026-01: RMB = 1.0% (88 values found)
2023-07: RMB = 1.0% (64 values found)
2024-07: RMB = 1.0% (80 values found)
2025-07: RMB = 1.0% (86 values found)
2023-06: RMB = 1.0% (63 values found)
2024-06: RMB = 1.0% (79 values found)
2025-06: RMB = 1.0% (85 values found)
2024-03: RMB = 1.0% (76 values found)
2025-03: RMB = 1.0% (88 values found)
2024-05: RMB = 1.0% (78 values found)
2025-05: RMB

In [ ]:
# Step 3: Preview the output
import pandas as pd

df = pd.read_csv(
    "/Users/psat0501/Desktop/HSG/Master Thesis/Thesis Data/SWIFT Dataset/SWIFT_Global_Currency_Shares.csv"
)
print(f"Shape: {df.shape}")
print(f"Date range: {df['Report_Month'].min()} to {df['Report_Month'].max()}")
df.head(10)